# 📊 Notebook 01 — Exploratory Data Analysis
**Customer Churn Prediction Project | Fall 2024**

Goals:
- Understand the dataset structure and distributions
- Identify correlations with churn
- Surface insights to guide feature engineering

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from data.loader import load_raw_data

sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.dpi'] = 120

df = load_raw_data()
print(f'Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

## 1. Dataset Overview

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Basic Stats ===')
df.describe(include='all').T

In [ ]:
# Churn rate
churn_rate = df['churn'].mean()
print(f'Overall Churn Rate: {churn_rate:.1%}')
print(f'Class Balance: {df["churn"].value_counts().to_dict()}')

fig, ax = plt.subplots(figsize=(5, 4))
counts = df['churn'].value_counts()
ax.pie(counts, labels=['Retained', 'Churned'], autopct='%1.1f%%',
       colors=['#2A9D8F', '#E63946'], startangle=90)
ax.set_title('Churn Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Numeric Feature Distributions

In [ ]:
numeric_cols = ['tenure', 'monthly_charges', 'total_charges', 'support_calls', 'num_products']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    ax = axes[i]
    df[df['churn']==0][col].hist(ax=ax, bins=30, alpha=0.6, color='#2A9D8F', label='Retained')
    df[df['churn']==1][col].hist(ax=ax, bins=30, alpha=0.6, color='#E63946', label='Churned')
    ax.set_title(col.replace('_', ' ').title(), fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)

axes[-1].set_visible(False)
plt.suptitle('Numeric Feature Distributions by Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Categorical Feature vs. Churn

In [ ]:
cat_cols = ['contract', 'payment_method', 'internet_service', 'gender', 'senior_citizen']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    ax = axes[i]
    churn_by_cat = df.groupby(col)['churn'].mean().sort_values(ascending=False)
    churn_by_cat.plot(kind='bar', ax=ax, color='#E63946', edgecolor='white')
    ax.set_title(f'Churn Rate by {col.replace("_", " ").title()}', fontweight='bold')
    ax.set_ylabel('Churn Rate')
    ax.set_ylim(0, 1)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    ax.tick_params(axis='x', rotation=30)
    ax.grid(axis='y', alpha=0.3)

axes[-1].set_visible(False)
plt.suptitle('Churn Rate by Categorical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Correlation Heatmap

In [ ]:
corr_cols = ['tenure', 'monthly_charges', 'total_charges', 'support_calls',
             'num_products', 'senior_citizen', 'churn']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Key Insights Summary

In [ ]:
print('=== KEY EDA FINDINGS ===')
print(f'1. Churn Rate: {df["churn"].mean():.1%} — class imbalance needs handling')
print(f'2. Month-to-month contract churn: {df[df["contract"]=="Month-to-month"]["churn"].mean():.1%}')
print(f'3. Two-year contract churn: {df[df["contract"]=="Two year"]["churn"].mean():.1%}')
print(f'4. Mean tenure (churned): {df[df["churn"]==1]["tenure"].mean():.1f} months')
print(f'5. Mean tenure (retained): {df[df["churn"]==0]["tenure"].mean():.1f} months')
print(f'6. Avg support calls (churned): {df[df["churn"]==1]["support_calls"].mean():.2f}')
print(f'7. Avg support calls (retained): {df[df["churn"]==0]["support_calls"].mean():.2f}')